In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score,mean_absolute_error, mean_squared_error
from transformers import BertTokenizerFast, BertModel
import os
from sklearn.model_selection import GroupShuffleSplit

# ========== 数据读取与特征 ==========
data = pd.read_excel("../invertebrates_EC10/invertebrates_EC10_unique.xlsx")
data = data.dropna()
smiles_data = data['SMILES_Canonical_RDKit'].tolist()
mgperL = data['mgperL'].values
Duration_Value = data['Duration_Value'].values

mgperL = np.log1p(mgperL)

# One-Hot 编码辅助
from sklearn.preprocessing import OneHotEncoder
def encode_column(df, column_name):
    unique_values = df[column_name].unique()
    if len(unique_values) > 1:
        enc = OneHotEncoder(sparse_output=False)
        return enc.fit_transform(df[[column_name]])
    else:
        return None

# 编码 effect、endpoint、species_group
effect_encoded   = encode_column(data, 'effect')
endpoint_encoded = encode_column(data, 'endpoint')
species_encoded  = encode_column(data, 'species_group')

# 拼接额外特征：Duration_Value + (one-hot们)
extra_features = data['Duration_Value'].values.reshape(-1, 1)
for enc in [effect_encoded, endpoint_encoded, species_encoded]:
    if enc is not None:
        extra_features = np.hstack((extra_features, enc))

extra_dim = extra_features.shape[1]
data_extra_features = extra_features.astype(np.float32)

# ========== 设备 ==========
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ========== 数据集 ==========
class SMILES_Dataset(Dataset):
    def __init__(self, smiles, reg_labels, extra_features=None, use_augmentation=False, tokenizer=None, max_length=128):
        self.smiles = smiles
        self.reg_labels = reg_labels
        self.extra_features = extra_features
        self.use_augmentation = use_augmentation
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        s = self.smiles[idx]
        reg_label = self.reg_labels[idx]
        tokens = self.tokenizer(s, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt")
        tokens = {key: val.squeeze(0) for key, val in tokens.items()}
        if self.extra_features is not None:
            extra_feat = self.extra_features[idx]
            extra_feat = torch.tensor(extra_feat, dtype=torch.float32)
            return tokens, torch.tensor(reg_label, dtype=torch.float32), extra_feat
        else:
            return tokens, torch.tensor(reg_label, dtype=torch.float32)

# ========== 加载本地 BERT ==========
checkpoint = '../models/base_bert'
tokenizer = BertTokenizerFast.from_pretrained(checkpoint)
bert_model = BertModel.from_pretrained(checkpoint)

# ========== 模型 ==========
class Bert_Regression(nn.Module):
    def __init__(self, dropout_rate, fc1_size, fc2_size, fc3_size, extra_dim=0):
        super(Bert_Regression, self).__init__()
        self.bert = bert_model
        hidden_size = self.bert.config.hidden_size
        input_dim = hidden_size + extra_dim
        self.regressor = nn.Sequential(
            nn.Linear(input_dim, fc1_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc1_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc1_size, fc2_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc2_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc2_size, fc3_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc3_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc3_size, 1),
            nn.Softplus()
        )

    def forward(self, tokens, extra_features=None):
        outputs = self.bert(**tokens)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        if extra_features is not None:
            x = torch.cat([cls_embedding, extra_features], dim=1)
        else:
            x = cls_embedding
        reg_output = self.regressor(x).squeeze(-1)
        return reg_output

# ========== 超参数（保持你给的 best_params） ==========
best_params = {
    'dropout_rate': 0.21357344690077285,
    'fc1_size': 896,
    'fc2_size': 352,
    'fc3_size': 192,
    'learning_rate': 1.885835543930883e-05,
    'weight_decay': 0.040884771694787714
}
best_dropout = best_params['dropout_rate']
best_fc1 = best_params['fc1_size']
best_fc2 = best_params['fc2_size']
best_fc3 = best_params['fc3_size']
best_lr = best_params['learning_rate']
best_wd = best_params['weight_decay']

# ========== 9:1 单次划分（按 SMILES 分组，避免泄漏） ==========
groups = np.array(smiles_data)
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(gss.split(smiles_data, mgperL, groups=groups))

train_smiles = [smiles_data[i] for i in train_idx]
val_smiles   = [smiles_data[i] for i in val_idx]
train_labels = mgperL[train_idx].astype(np.float32)
val_labels   = mgperL[val_idx].astype(np.float32)
train_extra  = data_extra_features[train_idx]
val_extra    = data_extra_features[val_idx]

train_dataset = SMILES_Dataset(train_smiles, train_labels, extra_features=train_extra, tokenizer=tokenizer)
val_dataset   = SMILES_Dataset(val_smiles,   val_labels,   extra_features=val_extra,   tokenizer=tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=4, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

# ========== 初始化模型 & 训练 ==========
model = Bert_Regression(best_dropout, best_fc1, best_fc2, best_fc3, extra_dim=extra_dim).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=best_wd)
criterion = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# 训练循环（不引入你说多余的早停变量）
for epoch in range(30):  # 如需改轮次可直接改这个常数，不新增变量
    model.train()
    train_losses = []
    for tokens, labels_tensor, extra_feats in train_loader:
        tokens = {k: v.to(device) for k, v in tokens.items()}
        labels_tensor = labels_tensor.to(device)
        extra_feats = extra_feats.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(tokens, extra_feats)
            loss = criterion(outputs, labels_tensor)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())

    # 验证
    model.eval()
    preds_all, targets_all = [], []
    with torch.no_grad():
        for tokens, labels_tensor, extra_feats in val_loader:
            tokens = {k: v.to(device) for k, v in tokens.items()}
            labels_tensor = labels_tensor.to(device)
            extra_feats = extra_feats.to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(tokens, extra_feats)
            preds_all.append(outputs.detach().cpu().numpy())
            targets_all.append(labels_tensor.detach().cpu().numpy())
    preds_all = np.concatenate(preds_all)
    targets_all = np.concatenate(targets_all)
    rmse = np.sqrt(np.mean((preds_all - targets_all) ** 2))
    mae  = mean_absolute_error(targets_all, preds_all)
    r2   = r2_score(targets_all, preds_all)
    print(f"Epoch {epoch+1:02d} | RMSE {rmse:.4f} | MAE {mae:.4f} | R2 {r2:.4f}")

# 保存训练后的模型
os.makedirs("./saved_models", exist_ok=True)
torch.save(model.state_dict(), "./saved_models/invertebrates_EC10.pth")
print("✅ 单次 9:1 训练完成，模型已保存到 ./saved_models/invertebrates_EC10.pth")

# ========== 提取嵌入并保存（与你原来逻辑一致） ==========
def get_embeddings_for_loader(model, loader):
    all_embeddings = []
    all_extra_features = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for tokens, labels_tensor, extra_feats in loader:
            tokens = {key: val.to(device) for key, val in tokens.items()}
            extra_feats = extra_feats.to(device)
            outputs = model.bert(**tokens)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [B, 768]
            all_embeddings.append(cls_embeddings.cpu().numpy())
            all_extra_features.append(extra_feats.cpu().numpy())
            all_labels.append(labels_tensor.numpy())
    return (
        np.vstack(all_embeddings),
        np.vstack(all_extra_features),
        np.concatenate(all_labels)
    )

# 保险起见可重载最佳权重，也可以直接用当前内存中的模型
# model.load_state_dict(torch.load("./saved_models/bert_reg_single_split.pth", map_location=device))

smiles_emb_train, extra_feat_train, labels_train = get_embeddings_for_loader(model, train_loader)
smiles_emb_val,   extra_feat_val,   labels_val   = get_embeddings_for_loader(model, val_loader)

X_train = np.hstack([smiles_emb_train, extra_feat_train])
X_val   = np.hstack([smiles_emb_val,   extra_feat_val])

os.makedirs('./invertebrates_EC10/embeddings', exist_ok=True)
os.makedirs('./invertebrates_EC10/bert_prediction', exist_ok=True)

np.save('./invertebrates_EC10/embeddings/train_single.npy', X_train)
np.save('./invertebrates_EC10/embeddings/val_single.npy',   X_val)
np.save('./invertebrates_EC10/embeddings/train_labels_single.npy', labels_train)
np.save('./invertebrates_EC10/embeddings/val_labels_single.npy',   labels_val)

# 验证集预测并保存（与你之前的保存方式一致）
y_preds = []
model.eval()
with torch.no_grad():
    for tokens, labels_tensor, extra_feats in val_loader:
        tokens = {k: v.to(device) for k, v in tokens.items()}
        extra_feats = extra_feats.to(device)
        outputs = model(tokens, extra_feats)
        y_preds.extend(outputs.squeeze().cpu().numpy())
y_preds = np.array(y_preds)
np.save('./invertebrates_EC10/bert_prediction/bert_single_val_predictions.npy', y_preds)

print("✅ 嵌入、标签与验证集预测已保存（single split 版本）。")


Epoch 01 | RMSE 1.7164 | MAE 1.2477 | R2 -0.0051
Epoch 02 | RMSE 1.5864 | MAE 1.1360 | R2 0.1413
Epoch 03 | RMSE 1.5697 | MAE 1.0952 | R2 0.1594
Epoch 04 | RMSE 1.5506 | MAE 1.1005 | R2 0.1797
Epoch 05 | RMSE 1.4925 | MAE 1.0558 | R2 0.2401
Epoch 06 | RMSE 1.4734 | MAE 1.0139 | R2 0.2593
Epoch 07 | RMSE 1.4546 | MAE 0.9842 | R2 0.2782
Epoch 08 | RMSE 1.4386 | MAE 0.9669 | R2 0.2939
Epoch 09 | RMSE 1.3854 | MAE 0.9267 | R2 0.3452
Epoch 10 | RMSE 1.3458 | MAE 0.9156 | R2 0.3820
Epoch 11 | RMSE 1.3163 | MAE 0.8708 | R2 0.4089
Epoch 12 | RMSE 1.3345 | MAE 0.8796 | R2 0.3924
Epoch 13 | RMSE 1.3277 | MAE 0.9007 | R2 0.3986
Epoch 14 | RMSE 1.3461 | MAE 0.8756 | R2 0.3818
Epoch 15 | RMSE 1.3594 | MAE 0.9238 | R2 0.3695
Epoch 16 | RMSE 1.3187 | MAE 0.8742 | R2 0.4067
Epoch 17 | RMSE 1.3101 | MAE 0.8642 | R2 0.4144
Epoch 18 | RMSE 1.3235 | MAE 0.8710 | R2 0.4024
Epoch 19 | RMSE 1.3541 | MAE 0.8719 | R2 0.3744
Epoch 20 | RMSE 1.3590 | MAE 0.8821 | R2 0.3699
Epoch 21 | RMSE 1.3112 | MAE 0.8612 | R

In [2]:
import os
import numpy as np
from xgboost import XGBRegressor

# 目录
data_dir = './invertebrates_EC10/embeddings'
save_dir = './invertebrates_EC10/xgb_predictions'
os.makedirs(save_dir, exist_ok=True)

# 最优超参数（保持不变）
best_params = {
    'n_estimators': 500,
    'learning_rate': 0.014509456908913889,
    'max_depth': 5,
    'subsample': 0.9387632330805947,
    'colsample_bytree': 0.6410965653177675
}


print("\n🟢 Processing single split...")

# 加载单次划分的数据（与前面保存的文件名对应）
X_train = np.load(os.path.join(data_dir, 'train_single.npy'))
y_train = np.load(os.path.join(data_dir, 'train_labels_single.npy'))
X_val   = np.load(os.path.join(data_dir, 'val_single.npy'))
y_val   = np.load(os.path.join(data_dir, 'val_labels_single.npy'))

# 初始化并训练模型
model = XGBRegressor(n_jobs=-1, verbosity=0, **best_params)
model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_val)

# 保存预测值和真实值（文件名改为 single 版本）
np.save(os.path.join(save_dir, 'single_y_pred.npy'), y_pred)
np.save(os.path.join(save_dir, 'single_y_true.npy'), y_val)

print("✅ Single split saved: single_y_pred.npy & single_y_true.npy")
print("\n🎉 XGBoost 单次划分预测结果已保存完毕！")




model_path = os.path.join(save_dir, 'xgb_single.json')
model.save_model(model_path)
print(f"💾 XGBoost 模型已保存至: {model_path}")



🟢 Processing single split...
✅ Single split saved: single_y_pred.npy & single_y_true.npy

🎉 XGBoost 单次划分预测结果已保存完毕！
💾 XGBoost 模型已保存至: ./invertebrates_EC10/xgb_predictions/xgb_single.json


In [3]:
# === 预测部分：固定 effect / duration（可选固定 endpoint/species），不重复定义类 ===
import os
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

# 路径配置（按需修改 NEW_EXCEL_PATH）
TRAIN_EXCEL_PATH = "../invertebrates_EC10/invertebrates_EC10_unique.xlsx"   # 训练用表（用于拟合 OneHot）
NEW_EXCEL_PATH   = "./Hazard-chemical-list.xlsx"                                    # ←← 你的新 SMILES Excel
BERT_WEIGHTS     = "./saved_models/invertebrates_EC10.pth"
XGB_MODEL_PATH   = "./invertebrates_EC10/xgb_predictions/xgb_single.json"
PRED_DIR         = "./predictions"
os.makedirs(PRED_DIR, exist_ok=True)

BATCH_SIZE = 256
MAX_LEN    = 128

# 1) 基于训练表拟合 OneHot，并输出 effect / Duration 信息
train_df = pd.read_excel(TRAIN_EXCEL_PATH).dropna(subset=[
    "SMILES_Canonical_RDKit","Duration_Value","effect","endpoint","species_group"
])

print("🔎 effect 类别及计数：")
print(train_df["effect"].value_counts().to_string())

print("\n🔎 Duration_Value 统计：")
print(train_df["Duration_Value"].describe())


🔎 effect 类别及计数：
effect
MOR    2256
ITX    2233
REP    1768
POP     789
DVP     313

🔎 Duration_Value 统计：
count     7359.000000
mean       310.989759
std        537.314784
min          0.000000
25%         48.000000
50%        120.000000
75%        504.000000
max      10794.000000
Name: Duration_Value, dtype: float64


In [4]:

enc_effect   = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(train_df[["effect"]])
enc_endpoint = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(train_df[["endpoint"]])
enc_species  = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(train_df[["species_group"]])

EXTRA_DIM = 1 + enc_effect.categories_[0].size + enc_endpoint.categories_[0].size + enc_species.categories_[0].size




In [7]:
BATCH_SIZE = 256
MAX_LEN    = 128

# ---------- 先读 checkpoint，推断训练时的 EXTRA_DIM ----------
state = torch.load(BERT_WEIGHTS, map_location=device)
w0 = state["regressor.0.weight"]          # [fc1_size, input_dim]
hidden = bert_model.config.hidden_size    # 一般 768
input_dim_ckpt = w0.shape[1]
EXTRA_DIM_CKPT = input_dim_ckpt - hidden
print(f"📏 从 checkpoint 推断的 EXTRA_DIM = {EXTRA_DIM_CKPT} (input_dim={input_dim_ckpt}, hidden={hidden})")

# ---------- 拟合 “可选” OneHot：仅当训练数据某列类别数>1 才加入 ----------
train_df = pd.read_excel(TRAIN_EXCEL_PATH).dropna(subset=[
    "SMILES_Canonical_RDKit","Duration_Value","effect","endpoint","species_group"
])

def fit_optional_ohe(df, col):
    vals = df[col].dropna().unique()
    if len(vals) > 1:
        enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        enc.fit(df[[col]])
        return enc
    return None

enc_effect   = fit_optional_ohe(train_df, "effect")
enc_endpoint = fit_optional_ohe(train_df, "endpoint")
enc_species  = fit_optional_ohe(train_df, "species_group")

# 统计训练时“理论”额外维度（按训练逻辑：单一类别就不加入one-hot）
extra_dim_by_train_logic = 1 \
    + (enc_effect.categories_[0].size   if enc_effect   else 0) \
    + (enc_endpoint.categories_[0].size if enc_endpoint else 0) \
    + (enc_species.categories_[0].size  if enc_species  else 0)
print(f"📐 按训练逻辑计算的 EXTRA_DIM = {extra_dim_by_train_logic}")

# ---------- 构建并加载 BERT_Regression（extra_dim 以 checkpoint 为准，确保加载不报错） ----------
bert_reg = Bert_Regression(best_dropout, best_fc1, best_fc2, best_fc3, extra_dim=EXTRA_DIM_CKPT).to(device)
bert_reg.load_state_dict(state, strict=True)
bert_reg.eval()

# ---------- 载入 XGBoost 模型 ----------
xgb_model = XGBRegressor()
xgb_model.load_model(XGB_MODEL_PATH)

# ---------- 读取新表（仅需 SMILES） ----------
raw = pd.read_excel(NEW_EXCEL_PATH).copy()
def pick_smiles(df):
    for n in ["SMILES_Canonical_RDKit","SMILES","smiles"]:
        if n in df.columns: return n
    return df.columns[0]  # 兜底
c_smiles = pick_smiles(raw)
smiles_list = raw[c_smiles].dropna().astype(str).tolist()
if len(smiles_list) == 0:
    raise ValueError("新表中未找到有效 SMILES。")



📏 从 checkpoint 推断的 EXTRA_DIM = 6 (input_dim=774, hidden=768)
📐 按训练逻辑计算的 EXTRA_DIM = 6


In [18]:
# ---------- 固定取值（在这里改） ----------
FIXED_EFFECT   = 'REP'           # 例如 "Mortality"；也可手动写字符串
FIXED_DURATION = float(672.0)  # 例如 48.0 或 96.0
FIXED_ENDPOINT = train_df["endpoint"].mode()[0]         # 例如 "EC10"
FIXED_SPECIES  = train_df["species_group"].mode()[0]    # 例如 "Invertebrates"

# ---------- 抽取 CLS768 ----------
cls_chunks = []
with torch.no_grad():
    for i in range(0, len(smiles_list), BATCH_SIZE):
        batch = smiles_list[i:i+BATCH_SIZE]
        toks = tokenizer(batch, padding="max_length", truncation=True,
                         max_length=MAX_LEN, return_tensors="pt")
        toks = {k: v.to(device) for k, v in toks.items()}
        out = bert_reg.bert(**toks)
        cls = out.last_hidden_state[:, 0, :].detach().cpu().numpy()
        cls_chunks.append(cls)
emb_cls = np.vstack(cls_chunks).astype(np.float32)   # [N, 768]

# ---------- 构造额外特征（只对训练中“参与 one-hot 的列”编码；不足则右侧零填充到 EXTRA_DIM_CKPT） ----------
n = emb_cls.shape[0]
parts = []

# duration
dur = np.full((n, 1), FIXED_DURATION, dtype=np.float32)
parts.append(dur)

# effect
if enc_effect is not None:
    eff = enc_effect.transform(pd.DataFrame({"effect": [FIXED_EFFECT]*n})).astype(np.float32)
    parts.append(eff)

# endpoint
if enc_endpoint is not None:
    endp = enc_endpoint.transform(pd.DataFrame({"endpoint": [FIXED_ENDPOINT]*n})).astype(np.float32)
    parts.append(endp)

# species
if enc_species is not None:
    spg = enc_species.transform(pd.DataFrame({"species_group": [FIXED_SPECIES]*n})).astype(np.float32)
    parts.append(spg)

extra_now = np.hstack(parts) if parts else np.zeros((n, 0), dtype=np.float32)
extra_cols = extra_now.shape[1]

# 右侧 pad 到 checkpoint 的 EXTRA_DIM
if extra_cols < EXTRA_DIM_CKPT:
    pad = np.zeros((n, EXTRA_DIM_CKPT - extra_cols), dtype=np.float32)
    extra_now = np.hstack([extra_now, pad])
elif extra_cols > EXTRA_DIM_CKPT:
    # 理论上不会发生（我们只在“参与过”的列上编码）；出现则截断到训练维度
    extra_now = extra_now[:, :EXTRA_DIM_CKPT]

print(f"✅ 额外特征维度: 当前 {extra_now.shape[1]}，与 checkpoint {EXTRA_DIM_CKPT} 对齐")

# ---------- 拼接并预测 ----------
X_new = np.hstack([emb_cls, extra_now]).astype(np.float32)  # [N, 768 + EXTRA_DIM_CKPT]
y_log = xgb_model.predict(X_new).astype(np.float32)
y_mgL = np.expm1(y_log)

out = pd.DataFrame({
    "SMILES": smiles_list,
    "effect_fixed":   FIXED_EFFECT,
    "duration_fixed": FIXED_DURATION,
    "endpoint_fixed": FIXED_ENDPOINT,
    "species_fixed":  FIXED_SPECIES,
    "pred_log_mgperL": y_log,
    "pred_mgperL":     y_mgL,
})
save_path = os.path.join(PRED_DIR, "invertebrates_EC10_REP_672.xlsx")
out.to_excel(save_path, index=False)
print(f"🎯 预测完成并保存：{save_path}")


✅ 额外特征维度: 当前 6，与 checkpoint 6 对齐
🎯 预测完成并保存：./predictions/invertebrates_EC10_REP_672.xlsx
